In [1]:
import torch
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn

In [2]:
transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [3]:
train_data = MNIST(root='./datasets/', train=True, download=False, transform = transform)
test_data = MNIST(root='./datasets/', train=False, download=False, transform = transform)

In [6]:
train_batch = DataLoader(dataset=train_data, batch_size = 2, shuffle=True)
test_batch = DataLoader(dataset=test_data, batch_size = 2, shuffle=True)

In [28]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_linear = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        self.fc_1 = nn.Linear(16*5*5, 120)
        self.fc_2 = nn.Linear(120, 84)
        self.fc_3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv_linear(x)
        x = self.fc_1(torch.flatten(x, 1))
        x = self.fc_2(x)
        x = self.fc_3(x)
        return x

In [29]:
model = LeNet()

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [31]:
Epochs = 5
size = len(train_batch.dataset)
for e in range(1, Epochs+1):
    correct = 0
    for idx, (x, y) in enumerate(train_batch):
        optimizer.zero_grad()
        predict = model(x)
        correct += (predict.argmax(1)==y).type(torch.float).sum().item()
        loss = criterion(predict, y)
        loss.backward()
        optimizer.step()
        if idx % 2000 == 0:
            print(f"Epoch: {e}     loss: {loss:>2.3f}")
    print(f"Acc: {(correct/size)*100:>2.3f}%")

Epoch: 1     loss: 2.247
Epoch: 1     loss: 1.995
Epoch: 1     loss: 0.668
Epoch: 1     loss: 0.024
Epoch: 1     loss: 0.046
Epoch: 1     loss: 0.122
Epoch: 1     loss: 0.002
Epoch: 1     loss: 0.205
Epoch: 1     loss: 0.817
Epoch: 1     loss: 0.214
Epoch: 1     loss: 0.046
Epoch: 1     loss: 0.011
Epoch: 1     loss: 0.005
Epoch: 1     loss: 0.076
Epoch: 1     loss: 0.051
Acc: 92.427%
Epoch: 2     loss: 0.014
Epoch: 2     loss: 0.050
Epoch: 2     loss: 0.001
Epoch: 2     loss: 0.001
Epoch: 2     loss: 0.017
Epoch: 2     loss: 0.001
Epoch: 2     loss: 0.003
Epoch: 2     loss: 0.018
Epoch: 2     loss: 0.000
Epoch: 2     loss: 0.015
Epoch: 2     loss: 0.001
Epoch: 2     loss: 0.000
Epoch: 2     loss: 0.000
Epoch: 2     loss: 0.008
Epoch: 2     loss: 0.000
Acc: 97.490%
Epoch: 3     loss: 0.191
Epoch: 3     loss: 0.002
Epoch: 3     loss: 0.000
Epoch: 3     loss: 0.000
Epoch: 3     loss: 0.000
Epoch: 3     loss: 0.000
Epoch: 3     loss: 0.020
Epoch: 3     loss: 0.024
Epoch: 3     loss: 0.000

In [37]:
model.eval()
correct = 0
size = len(test_batch.dataset)
with torch.no_grad():
    for x, y in test_batch:
        predict = model(x)
        correct += (predict.argmax(1)==y).type(torch.float).sum().item()
print(f"Acc: {(correct/size)*100:>2.3f}%")

Acc: 98.780%


In [38]:
torch.save(model.state_dict(), "model_lenet.pth")